610 manually annotated images

        ↓

Train YOLO model

        ↓

Use model inside CVAT

        ↓

Auto-label remaining images

        ↓
        
Only FIX mistakes (10x faster) Human Touch


In [ ]:
!nvidia-smi

Fri Feb 27 16:05:41 2026       
+-----------------------------------------------------------------------------------------+
| NVIDIA-SMI 580.82.07              Driver Version: 580.82.07      CUDA Version: 13.0     |
+-----------------------------------------+------------------------+----------------------+
| GPU  Name                 Persistence-M | Bus-Id          Disp.A | Volatile Uncorr. ECC |
| Fan  Temp   Perf          Pwr:Usage/Cap |           Memory-Usage | GPU-Util  Compute M. |
|                                         |                        |               MIG M. |
|=========================================+========================+======================|
|   0  Tesla T4                       Off |   00000000:00:04.0 Off |                    0 |
| N/A   43C    P8             11W /   70W |       0MiB /  15360MiB |      0%      Default |
|                                         |                        |                  N/A |
+-----------------------------------------+-----

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [ ]:
!cp /content/drive/MyDrive/SafeCommuteAI/dataset.zip /content/
!unzip -q /content/dataset.zip -d /content/
!echo "Train images:" && ls /content/dataset/annotated/images/train/ | wc -l
!echo "Val images:"   && ls /content/dataset/annotated/images/val/   | wc -l

Train images:
488
Val images:
122


Cell 4 — Install YOLOv8

In [ ]:
!pip install ultralytics -q
import ultralytics
ultralytics.checks()

Ultralytics 8.4.18 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Setup complete ✅ (2 CPUs, 12.7 GB RAM, 44.2/112.6 GB disk)


Cell 5 — Fix dataset.yaml


In [ ]:
import yaml

config = {
    "path": "/content/dataset/annotated",
    "train": "images/train",
    "val":   "images/val",
    "nc": 1,
    "names": {0: "bus"}
}

with open("/content/dataset/annotated/dataset.yaml", "w") as f:
    yaml.dump(config, f, default_flow_style=False)

print("Done:")
!cat /content/dataset/annotated/dataset.yaml

Done:
names:
  0: bus
nc: 1
path: /content/dataset/annotated
train: images/train
val: images/val


In [ ]:
from pathlib import Path

issues = []
for txt in Path("/content/dataset/annotated/labels/train").glob("*.txt"):
    for line in txt.read_text().strip().splitlines():
        parts = line.strip().split()
        if len(parts) != 5:
            issues.append(f"{txt.name}: malformed → {line}")
        elif int(parts[0]) != 0:
            issues.append(f"{txt.name}: unexpected class {parts[0]}")

if issues:
    print(f"Issues: {len(issues)}")
    for i in issues[:20]: print(i)
else:
    print("All labels valid. Only class 0 (bus) found. Ready to train.")

All labels valid. Only class 0 (bus) found. Ready to train.


In [ ]:
!yolo task=detect \
      mode=train \
      model=yolov8m.pt \
      data=/content/dataset/annotated/dataset.yaml \
      epochs=100 \
      imgsz=640 \
      batch=16 \
      patience=20 \
      name=safecommute_v1 \
      project=/content/drive/MyDrive/SafeCommuteAI/runs \
      device=0 \
      exist_ok=True \
      cos_lr=True \
      mosaic=1.0

Ultralytics 8.4.18 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/dataset/annotated/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=100, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int8=False, iou=0.7, keras=False, kobj=1.0, line_width=None, lr0=0.01, lrf=0.01, mask_ratio=4, max_det=300, mixup=0.0, mode=train, model=yolov8m.pt, momentum=0.937, mosaic=1.0, multi_scale=0.0, name=safecommute_v1, nbs=64, nms=False, opset=None, optimize=False, optimizer=auto, overlap_mask=True, patience=2

In [ ]:
from ultralytics import YOLO
from pathlib import Path

# Load your best model
model = YOLO("/content/drive/MyDrive/SafeCommuteAI/runs/safecommute_v1/weights/best.pt")

UNANNO_IMG   = Path("/content/dataset/unannotated/images")
OUTPUT_LABELS = Path("/content/dataset/unannotated/labels")
OUTPUT_LABELS.mkdir(parents=True, exist_ok=True)

results = model.predict(
    source=str(UNANNO_IMG),
    conf=0.35,
    iou=0.45,
    imgsz=640,
    save=False,
    stream=True
)

count = 0
for result in results:
    img_path = Path(result.path)
    label_path = OUTPUT_LABELS / img_path.with_suffix(".txt").name

    with open(label_path, "w") as f:
        if result.boxes is not None and len(result.boxes):
            for box in result.boxes:
                cls  = int(box.cls[0])
                xywh = box.xywhn[0].tolist()
                f.write(f"{cls} {xywh[0]:.6f} {xywh[1]:.6f} {xywh[2]:.6f} {xywh[3]:.6f}\n")
    count += 1

print(f"Done. Labels generated for {count} images.")
print(f"Saved to: {OUTPUT_LABELS}")


image 1/548 /content/dataset/unannotated/images/1614.jpg: 448x640 2 buss, 162.5ms
image 2/548 /content/dataset/unannotated/images/1615.jpg: 640x448 1 bus, 153.1ms
image 3/548 /content/dataset/unannotated/images/1616.jpg: 384x640 1 bus, 109.4ms
image 4/548 /content/dataset/unannotated/images/1617.jpg: 480x640 2 buss, 96.0ms
image 5/548 /content/dataset/unannotated/images/1618.jpg: 640x480 1 bus, 44.6ms
image 6/548 /content/dataset/unannotated/images/1619.jpg: 640x384 1 bus, 45.2ms
image 7/548 /content/dataset/unannotated/images/1620.jpg: 640x480 3 buss, 28.1ms
image 8/548 /content/dataset/unannotated/images/1621.jpg: 640x576 2 buss, 44.6ms
image 9/548 /content/dataset/unannotated/images/1622.jpg: 480x640 1 bus, 28.2ms
image 10/548 /content/dataset/unannotated/images/1623.jpg: 640x480 1 bus, 28.3ms
image 11/548 /content/dataset/unannotated/images/1624.jpg: 640x512 2 buss, 45.7ms
image 12/548 /content/dataset/unannotated/images/1625.jpg: 640x512 1 bus, 20.4ms
image 13/548 /content/datase

In [ ]:
!ls /content/dataset/unannotated/images/ | wc -l

548


In [ ]:
from pathlib import Path

LBL_DIR = Path("/content/dataset/unannotated/labels")
all_labels = list(LBL_DIR.glob("*.txt"))

empty    = [f for f in all_labels if f.stat().st_size == 0]
nonempty = [f for f in all_labels if f.stat().st_size > 0]

print(f"Total label files:     {len(all_labels)}")
print(f"With detections:       {len(nonempty)}")
print(f"Empty (no bus found):  {len(empty)}")

# Show box count distribution
from collections import Counter
box_counts = Counter()
for f in nonempty:
    lines = f.read_text().strip().splitlines()
    box_counts[len(lines)] += 1

print("\nBoxes per image distribution:")
for count, freq in sorted(box_counts.items()):
    print(f"  {count} box(es): {freq} images")

Total label files:     548
With detections:       543
Empty (no bus found):  5

Boxes per image distribution:
  1 box(es): 373 images
  2 box(es): 118 images
  3 box(es): 35 images
  4 box(es): 6 images
  5 box(es): 7 images
  6 box(es): 2 images
  7 box(es): 2 images


In [ ]:
import shutil, zipfile
from pathlib import Path

IMG_DIR = Path("/content/dataset/unannotated/images")
LBL_DIR = Path("/content/dataset/unannotated/labels")
PKG_DIR = Path("/content/cvat_package/obj_train_data")
PKG_DIR.mkdir(parents=True, exist_ok=True)

# Copy images + labels
for img in IMG_DIR.glob("*.jpg"):
    shutil.copy(img, PKG_DIR / img.name)
    lbl = LBL_DIR / img.with_suffix(".txt").name
    if lbl.exists():
        shutil.copy(lbl, PKG_DIR / lbl.name)

# obj.names
Path("/content/cvat_package/obj.names").write_text("bus\n")

# obj.data
Path("/content/cvat_package/obj.data").write_text(
    "classes = 1\ntrain = data/train.txt\nnames = data/obj.names\nbackup = backup/\n"
)

# train.txt
with open("/content/cvat_package/train.txt", "w") as f:
    for img in IMG_DIR.glob("*.jpg"):
        f.write(f"data/obj_train_data/{img.name}\n")

# Zip
with zipfile.ZipFile("/content/cvat_predictions.zip", "w") as z:
    for fp in Path("/content/cvat_package").rglob("*"):
        z.write(fp, fp.relative_to("/content/cvat_package"))

print("cvat_predictions.zip ready.")
!ls -lh /content/cvat_predictions.zip

cvat_predictions.zip ready.
-rw-r--r-- 1 root root 120M Feb 27 16:47 /content/cvat_predictions.zip


In [ ]:
from google.colab import files
files.download("/content/cvat_predictions.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

merger

In [ ]:
import shutil, zipfile
from pathlib import Path

# Sources (already in Colab)
GOOD_IMG   = Path("/content/dataset/annotated/images")
GOOD_LBL   = Path("/content/dataset/annotated/labels")
UNANNO_IMG = Path("/content/dataset/unannotated/images")
UNANNO_LBL = Path("/content/dataset/unannotated/labels")

PKG_DIR = Path("/content/cvat_complete/obj_train_data")
PKG_DIR.mkdir(parents=True, exist_ok=True)

# Copy 610 good annotations
for img in GOOD_IMG.rglob("*.jpg"):
    shutil.copy(img, PKG_DIR / img.name)
for lbl in GOOD_LBL.rglob("*.txt"):
    shutil.copy(lbl, PKG_DIR / lbl.name)

# Copy 548 auto-annotated
for img in UNANNO_IMG.glob("*.jpg"):
    shutil.copy(img, PKG_DIR / img.name)
    lbl = UNANNO_LBL / img.with_suffix(".txt").name
    if lbl.exists():
        shutil.copy(lbl, PKG_DIR / lbl.name)

# obj.names
Path("/content/cvat_complete/obj.names").write_text("bus\n")

# obj.data
Path("/content/cvat_complete/obj.data").write_text(
    "classes = 1\ntrain = data/train.txt\nnames = data/obj.names\nbackup = backup/\n"
)

# train.txt
with open("/content/cvat_complete/train.txt", "w") as f:
    for img in PKG_DIR.glob("*.jpg"):
        f.write(f"data/obj_train_data/{img.name}\n")

# Zip
with zipfile.ZipFile("/content/cvat_complete.zip", "w") as z:
    for fp in Path("/content/cvat_complete").rglob("*"):
        z.write(fp, fp.relative_to("/content/cvat_complete"))

print(f"Images packed:  {len(list(PKG_DIR.glob('*.jpg')))}")
print(f"Labels packed:  {len(list(PKG_DIR.glob('*.txt')))}")
print("cvat_complete.zip ready.")

Images packed:  1158
Labels packed:  1158
cvat_complete.zip ready.


In [ ]:
from google.colab import files
files.download("/content/cvat_complete.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

### **RETRAINIG MODEL WITH UPDATED DATASET**

In [ ]:
# Cell 1 - Setup
!pip install ultralytics -q
import torch
print(f"GPU: {torch.cuda.get_device_name(0)}")

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.2/1.2 MB 38.6 MB/s eta 0:00:00
GPU: Tesla T4


In [ ]:
# Cell 2 - Extract

import zipfile, os
from pathlib import Path

ZIP = "/content/drive/MyDrive/BusesDataset/Model01/batch01-model1.zip"
EXTRACT = "/content/model1_data"

with zipfile.ZipFile(ZIP, 'r') as z:
    z.extractall(EXTRACT)

# Find images and labels
imgs = list(Path(EXTRACT).rglob("*.jpg"))
print(f"Total images: {len(imgs)}")

Total images: 1156


In [ ]:
# Cell 3 - Build dataset with train/val split
import shutil, random
from pathlib import Path

SRC = Path(EXTRACT)
DATASET = Path("/content/buses_final")

for split in ["train", "val"]:
    (DATASET / "images" / split).mkdir(parents=True, exist_ok=True)
    (DATASET / "labels" / split).mkdir(parents=True, exist_ok=True)

all_imgs = sorted(SRC.rglob("*.jpg"))
random.seed(42)
random.shuffle(all_imgs)

split_idx = int(len(all_imgs) * 0.85)
train_imgs = all_imgs[:split_idx]
val_imgs   = all_imgs[split_idx:]

for img in train_imgs:
    lbl = img.with_suffix(".txt")
    shutil.copy(img, DATASET / "images" / "train" / img.name)
    if lbl.exists():
        shutil.copy(lbl, DATASET / "labels" / "train" / lbl.name)

for img in val_imgs:
    lbl = img.with_suffix(".txt")
    shutil.copy(img, DATASET / "images" / "val" / img.name)
    if lbl.exists():
        shutil.copy(lbl, DATASET / "labels" / "val" / lbl.name)

print(f"Train: {len(train_imgs)} | Val: {len(val_imgs)}")

Train: 982 | Val: 174


In [ ]:
# Cell 4 - dataset.yaml
import yaml

config = {
    "path": "/content/buses_final",
    "train": "images/train",
    "val":   "images/val",
    "nc": 1,
    "names": {0: "bus"}
}

with open("/content/buses_final/dataset.yaml", "w") as f:
    yaml.dump(config, f, default_flow_style=False)

print("dataset.yaml ready!!!")

dataset.yaml ready!!!


In [ ]:
# Retrain from best.pt

!yolo task=detect \
      mode=train \
      model=yolov8m.pt \
      data=/content/buses_final/dataset.yaml \
      epochs=50 \
      imgsz=640 \
      batch=16 \
      patience=15 \
      name=buses_model1_v2 \
      project=/content/drive/MyDrive/SafeCommuteAI/runs \
      device=0 \
      exist_ok=True \
      cos_lr=True \
      mosaic=1.0

Creating new Ultralytics Settings v0.0.6 file ✅ 
View Ultralytics Settings with 'yolo settings' or at '/root/.config/Ultralytics/settings.json'
Update Settings with 'yolo settings key=value', i.e. 'yolo settings runs_dir=path/to/dir'. For help see https://docs.ultralytics.com/quickstart/#ultralytics-settings.
Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
engine/trainer: agnostic_nms=False, amp=True, angle=1.0, augment=False, auto_augment=randaugment, batch=16, bgr=0.0, box=7.5, cache=False, cfg=None, classes=None, close_mosaic=10, cls=0.5, compile=False, conf=None, copy_paste=0.0, copy_paste_mode=flip, cos_lr=True, cutmix=0.0, data=/content/buses_final/dataset.yaml, degrees=0.0, deterministic=True, device=0, dfl=1.5, dnn=False, dropout=0.0, dynamic=False, embed=None, end2end=None, epochs=50, erasing=0.4, exist_ok=True, fliplr=0.5, flipud=0.0, format=torchscript, fraction=1.0, freeze=None, half=False, hsv_h=0.015, hsv_s=0.7, hsv_v=0.4, imgsz=640, int

In [ ]:
# Cell 6 - Results
from ultralytics import YOLO
model = YOLO("/content/drive/MyDrive/SafeCommuteAI/runs/buses_model1_v2/weights/best.pt")
metrics = model.val(data="/content/buses_final/dataset.yaml")
print(f"mAP50:    {metrics.box.map50:.3f}")
print(f"mAP50-95: {metrics.box.map:.3f}")
print(f"Precision:{metrics.box.mp:.3f}")
print(f"Recall:   {metrics.box.mr:.3f}")

Ultralytics 8.4.19 🚀 Python-3.12.12 torch-2.10.0+cu128 CUDA:0 (Tesla T4, 14913MiB)
Model summary (fused): 93 layers, 25,840,339 parameters, 0 gradients, 78.7 GFLOPs
val: Fast image access ✅ (ping: 0.0±0.0 ms, read: 3399.9±1207.7 MB/s, size: 220.6 KB)
val: Scanning /content/buses_final/labels/val.cache... 174 images, 1 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 174/174 16.6Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 100% ━━━━━━━━━━━━ 11/11 2.2it/s 5.0s
                   all        174        229      0.939      0.938      0.976      0.888
Speed: 3.4ms preprocess, 15.9ms inference, 0.0ms loss, 1.8ms postprocess per image
Results saved to /content/runs/detect/val
mAP50:    0.976
mAP50-95: 0.888
Precision:0.939
Recall:   0.938


### FIXING NEW DATASET (not for model1 model3 batch03 dataset fixed)

In [ ]:
# Upload your CVAT zip
uploaded = files.upload()
ZIP_NAME = list(uploaded.keys())[0]
print(f"Uploaded: {ZIP_NAME}")

Saving job_12_annotations_2026_03_03_08_10_35_yolo 1.1.zip to job_12_annotations_2026_03_03_08_10_35_yolo 1.1.zip
✅ Uploaded: job_12_annotations_2026_03_03_08_10_35_yolo 1.1.zip


In [ ]:
import zipfile, os, shutil
from pathlib import Path

CVAT_ZIP   = f"/content/{ZIP_NAME}"
CVAT_DIR   = Path("/content/cvat_annotations")
DRIVE_IMGS = Path("/content/drive/MyDrive/BusesDataset/Model02/Batch3/NOT")
OUTPUT_DIR = Path("/content/dataset_ready")
IMG_OUT    = OUTPUT_DIR / "obj_train_data"
IMG_OUT.mkdir(parents=True, exist_ok=True)

# Extract CVAT zip
with zipfile.ZipFile(CVAT_ZIP, 'r') as z:
    z.extractall(CVAT_DIR)

print("CVAT contents:")
for f in CVAT_DIR.rglob("*"):
    print(f" {f.relative_to(CVAT_DIR)}")

CVAT contents:
 train.txt
 obj_train_data
 obj.data
 obj.names
 obj_train_data/NOT
 obj_train_data/NOT/0942_bus_2.txt
 obj_train_data/NOT/01983_bus_0.txt
 obj_train_data/NOT/01148_bus_0.txt
 obj_train_data/NOT/01387_bus_0.txt
 obj_train_data/NOT/01759_bus_0.txt
 obj_train_data/NOT/01520_bus_0.txt
 obj_train_data/NOT/01514_bus_0.txt
 obj_train_data/NOT/0889_bus_1.txt
 obj_train_data/NOT/01860_bus_0.txt
 obj_train_data/NOT/01873_bus_0.txt
 obj_train_data/NOT/01687_bus_0.txt
 obj_train_data/NOT/01390_bus_0.txt
 obj_train_data/NOT/02029_bus_0.txt
 obj_train_data/NOT/01146_bus_0.txt
 obj_train_data/NOT/0850_bus_0.txt
 obj_train_data/NOT/01543_bus_0.txt
 obj_train_data/NOT/01619_bus_0.txt
 obj_train_data/NOT/01367_bus_0.txt
 obj_train_data/NOT/01177_bus_0.txt
 obj_train_data/NOT/01885_bus_0.txt
 obj_train_data/NOT/0972_bus_0.txt
 obj_train_data/NOT/01204_bus_0.txt
 obj_train_data/NOT/02110_bus_0.txt
 obj_train_data/NOT/01645_bus_0.txt
 obj_train_data/NOT/0984_bus_0.txt
 obj_train_data/NOT/01

In [ ]:
import shutil, os
from pathlib import Path

CVAT_DIR   = Path("/content/cvat_annotations")
DRIVE_IMGS = Path("/content/drive/MyDrive/BusesDataset/Model02/Batch3/NOT")
OUTPUT_DIR = Path("/content/dataset_ready")
IMG_OUT    = OUTPUT_DIR / "obj_train_data"
IMG_OUT.mkdir(parents=True, exist_ok=True)

# ✅ Correct path — labels are inside obj_train_data/NOT/
lbl_src = CVAT_DIR / "obj_train_data" / "NOT"
label_files = list(lbl_src.glob("*.txt"))

print(f"Labels in CVAT    : {len(label_files)}")
print(f"Images in Drive   : {len(list(DRIVE_IMGS.glob('*.jpg')))}")

matched  = 0
missing  = 0

for lbl in label_files:
    img_path = DRIVE_IMGS / f"{lbl.stem}.jpg"
    if not img_path.exists():
        missing += 1
        print(f"  ❌ No image: {lbl.stem}.jpg — skipped")
        continue
    shutil.copy(img_path, IMG_OUT / f"{lbl.stem}.jpg")
    shutil.copy(lbl,      IMG_OUT / f"{lbl.stem}.txt")
    matched += 1

# Copy meta files
for fname in ["obj.names", "obj.data"]:
    src = CVAT_DIR / fname
    if src.exists():
        shutil.copy(src, OUTPUT_DIR / fname)

# Regenerate clean train.txt
images = sorted(IMG_OUT.glob("*.jpg"))
with open(OUTPUT_DIR / "train.txt", "w") as f:
    for img in images:
        f.write(f"obj_train_data/{img.name}\n")

# Stats
valid_lbls = sum(1 for l in IMG_OUT.glob("*.txt") if l.stat().st_size > 0)
empty_lbls = sum(1 for l in IMG_OUT.glob("*.txt") if l.stat().st_size == 0)

print(f"\n{'='*50}")
print(f"✅ Matched & copied : {matched}")
print(f"❌ Missing images   : {missing}")
print(f"   Valid labels     : {valid_lbls}")
print(f"   Empty labels     : {empty_lbls}")
print(f"   Total images     : {len(images)}")
print(f"{'='*50}")

Labels in CVAT    : 1119
Images in Drive   : 1544

✅ Matched & copied : 1119
❌ Missing images   : 0
   Valid labels     : 993
   Empty labels     : 126
   Total images     : 1119


In [ ]:
import shutil
from google.colab import files

shutil.make_archive("/content/dataset_ready_clean", "zip", OUTPUT_DIR)
print(f"Size: {round(os.path.getsize('/content/dataset_ready_clean.zip')/1e6, 1)} MB")
files.download("/content/dataset_ready_clean.zip")

Size: 384.5 MB


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>